In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Locate project root
project_root = Path.cwd()

if not (project_root / ".env").exists():
    project_root = project_root.parent

# Load environment variables
load_dotenv(project_root / ".env")

# Verify API key
api_key = os.getenv("GROQ_API_KEY")

if api_key:
    print("Groq API key loaded successfully!")
    print("Project root:", project_root)
else:
    print("Groq API key not found. Check your .env file.")

Groq API key loaded successfully!
Project root: d:\Fluid AI project\Autonomous-AI-Agent


In [2]:
import os
from groq import Groq
from dotenv import load_dotenv

# Load API key
load_dotenv(project_root / ".env")

# Initialize Groq client
client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

# Retrieve available models
models = client.models.list()

print("Groq API connected successfully!")
print("\nAvailable models:")

for model in models.data:
    print(model.id)

Groq API connected successfully!

Available models:
openai/gpt-oss-20b
whisper-large-v3-turbo
allam-2-7b
canopylabs/orpheus-v1-english
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-120b
canopylabs/orpheus-arabic-saudi
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-safeguard-20b
whisper-large-v3
qwen/qwen3.8-27b


In [3]:
MODEL_NAME = "openai/gpt-oss-120b"

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "system",
            "content": "You are an intelligent AI planning assistant."
        },
        {
            "role": "user",
            "content": """
            Create a project proposal for an AI-powered
            student attendance management system.

            Identify the major tasks required to create
            this proposal. Return a numbered task list.
            """
        }
    ],
    temperature=0.3,
    max_completion_tokens=1000
)

print(response.choices[0].message.content)

**Major Tasks for Preparing the Project Proposal – AI‑Powered Student Attendance Management System**

1. **Define Project Vision & Objectives**  
   - Articulate the problem statement and desired outcomes (e.g., real‑time attendance tracking, reduced manual effort, early‑warning alerts).  
   - Align objectives with institutional goals (student safety, academic analytics, compliance).

2. **Stakeholder Identification & Requirements Gathering**  
   - List primary users (students, teachers, administrators) and secondary users (parents, IT staff).  
   - Conduct interviews / surveys to capture functional (face‑recognition, RFID, mobile check‑in) and non‑functional (privacy, accuracy, latency) requirements.

3. **Market & Competitive Analysis**  
   - Research existing attendance solutions (hardware‑only, software‑only, AI‑enabled).  
   - Identify gaps, pricing models, and differentiators for the proposed system.

4. **Regulatory & Privacy Assessment**  
   - Review applicable data‑prote

In [4]:
from pydantic import BaseModel, Field
from typing import List


class Task(BaseModel):
    task_id: int
    title: str
    description: str
    status: str = "pending"


class AgentPlan(BaseModel):
    document_type: str
    document_title: str
    tasks: List[Task]

In [5]:
import json

planning_prompt = """
You are an autonomous AI project planning agent.

Analyze the user's request and create an execution plan.

Return ONLY valid JSON using this structure:

{
    "document_type": "Project Proposal",
    "document_title": "AI Student Attendance System",
    "tasks": [
        {
            "task_id": 1,
            "title": "Define project objectives",
            "description": "Identify the main project objectives",
            "status": "pending"
        }
    ]
}

Create 5 to 10 meaningful tasks.
Tasks must be specific and executable.
"""

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": planning_prompt},
        {
            "role": "user",
            "content": "Create a proposal for an AI-powered student attendance system."
        }
    ],
    temperature=0.2,
    max_completion_tokens=2000
)

raw_plan = response.choices[0].message.content

print(raw_plan)

{
    "document_type": "Project Proposal",
    "document_title": "AI Student Attendance System",
    "tasks": [
        {
            "task_id": 1,
            "title": "Define project objectives",
            "description": "Identify the primary goals, success criteria, and expected outcomes for the AI-powered attendance system.",
            "status": "pending"
        },
        {
            "task_id": 2,
            "title": "Conduct stakeholder analysis",
            "description": "Gather requirements from teachers, administrators, students, and IT staff to ensure the system meets all user needs.",
            "status": "pending"
        },
        {
            "task_id": 3,
            "title": "Design system architecture",
            "description": "Create a high‑level diagram covering data flow, AI model integration, hardware components, and cloud/on‑premise deployment options.",
            "status": "pending"
        },
        {
            "task_id": 4,
            "tit

In [7]:
plan_data = json.loads(raw_plan)

agent_plan = AgentPlan.model_validate(plan_data)

print("Document:", agent_plan.document_title)

for task in agent_plan.tasks:
    print(f"{task.task_id}. {task.title} [{task.status}]")

Document: AI Student Attendance System
1. Define project objectives [pending]
2. Conduct stakeholder analysis [pending]
3. Design system architecture [pending]
4. Develop AI model for facial recognition [pending]
5. Integrate hardware and software components [pending]
6. Implement privacy and data security measures [pending]
7. Develop user interfaces [pending]
8. Conduct pilot testing [pending]
9. Evaluate performance and refine system [pending]
10. Prepare deployment and training plan [pending]


In [8]:
from typing import Optional


class TaskResult(BaseModel):
    task_id: int
    title: str
    status: str
    content: Optional[str] = None
    error: Optional[str] = None

In [9]:
def execute_task(task, document_title):

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are an expert technical project proposal writer. "
                        "Generate clear, detailed, and professional content "
                        "for the assigned project section."
                    )
                },
                {
                    "role": "user",
                    "content": f"""
                    Project: {document_title}

                    Section: {task.title}

                    Task Description: {task.description}

                    Write professional, relevant content for this section.
                    Use headings and bullet points wherever appropriate.
                    """
                }
            ],
            temperature=0.3,
            max_completion_tokens=1200
        )

        content = response.choices[0].message.content

        task.status = "completed"

        return TaskResult(
            task_id=task.task_id,
            title=task.title,
            status="completed",
            content=content
        )

    except Exception as e:

        task.status = "failed"

        return TaskResult(
            task_id=task.task_id,
            title=task.title,
            status="failed",
            error=str(e)
        )

In [10]:
result = execute_task(
    agent_plan.tasks[0],
    agent_plan.document_title
)

print("Task:", result.title)
print("Status:", result.status)
print("\nGenerated Content:\n")
print(result.content)

Task: Define project objectives
Status: completed

Generated Content:

## 1. Project Objectives  

The **AI‑Powered Student Attendance System** is intended to replace manual roll‑call processes with an automated, reliable, and privacy‑respecting solution that leverages computer‑vision and machine‑learning technologies. The objectives below articulate what the project will achieve, how success will be measured, and the tangible benefits expected for all stakeholders.

---

### 1.1 Primary Goals  

| # | Goal | Description | Rationale |
|---|------|-------------|-----------|
| **G1** | **Automated, real‑time attendance capture** | Use AI‑driven facial recognition (or alternative biometric modalities) to record student presence the moment they enter a classroom or virtual session. | Eliminates teacher workload, reduces human error, and provides instantaneous data. |
| **G2** | **High accuracy and reliability** | Achieve ≥ 95 % identification accuracy and ≤ 2 % false‑negative/false‑positiv

In [11]:
def execute_all_tasks(agent_plan):

    results = []

    for task in agent_plan.tasks:

        print(f"Executing Task {task.task_id}: {task.title}")

        result = execute_task(
            task,
            agent_plan.document_title
        )

        results.append(result)

        print(f"Status: {result.status}")
        print("-" * 60)

    return results

In [12]:
task_results = execute_all_tasks(agent_plan)

Executing Task 1: Define project objectives
Status: completed
------------------------------------------------------------
Executing Task 2: Conduct stakeholder analysis
Status: completed
------------------------------------------------------------
Executing Task 3: Design system architecture
Status: completed
------------------------------------------------------------
Executing Task 4: Develop AI model for facial recognition
Status: completed
------------------------------------------------------------
Executing Task 5: Integrate hardware and software components
Status: completed
------------------------------------------------------------
Executing Task 6: Implement privacy and data security measures
Status: completed
------------------------------------------------------------
Executing Task 7: Develop user interfaces
Status: completed
------------------------------------------------------------
Executing Task 8: Conduct pilot testing
Status: completed
-----------------------------

In [13]:
for result in task_results:

    print(f"Task ID: {result.task_id}")
    print(f"Title: {result.title}")
    print(f"Status: {result.status}")

    if result.content:
        print(f"Content Length: {len(result.content)} characters")
    else:
        print("Content: Missing")

    print("-" * 50)

Task ID: 1
Title: Define project objectives
Status: completed
Content Length: 5008 characters
--------------------------------------------------
Task ID: 2
Title: Conduct stakeholder analysis
Status: completed
Content Length: 5474 characters
--------------------------------------------------
Task ID: 3
Title: Design system architecture
Status: completed
Content Length: 4175 characters
--------------------------------------------------
Task ID: 4
Title: Develop AI model for facial recognition
Status: completed
Content Length: 4585 characters
--------------------------------------------------
Task ID: 5
Title: Integrate hardware and software components
Status: completed
Content Length: 4218 characters
--------------------------------------------------
Task ID: 6
Title: Implement privacy and data security measures
Status: completed
Content Length: 4910 characters
--------------------------------------------------
Task ID: 7
Title: Develop user interfaces
Status: completed
Content Length: 

In [14]:
def validate_task_results(task_results):

    validation_report = []

    for result in task_results:

        issues = []

        if result.status != "completed":
            issues.append("Task execution failed")

        if not result.content:
            issues.append("Generated content is empty")

        elif len(result.content.strip()) < 300:
            issues.append("Generated content is too short")

        validation_report.append({
            "task_id": result.task_id,
            "title": result.title,
            "status": result.status,
            "content_length": len(result.content or ""),
            "passed": len(issues) == 0,
            "issues": issues
        })

    return validation_report

In [15]:
validation_report = validate_task_results(task_results)

for item in validation_report:

    print(f"{item['task_id']}. {item['title']}")
    print("Validation:", "PASSED" if item["passed"] else "FAILED")
    print("Issues:", item["issues"])
    print("-" * 50)

1. Define project objectives
Validation: PASSED
Issues: []
--------------------------------------------------
2. Conduct stakeholder analysis
Validation: PASSED
Issues: []
--------------------------------------------------
3. Design system architecture
Validation: PASSED
Issues: []
--------------------------------------------------
4. Develop AI model for facial recognition
Validation: PASSED
Issues: []
--------------------------------------------------
5. Integrate hardware and software components
Validation: PASSED
Issues: []
--------------------------------------------------
6. Implement privacy and data security measures
Validation: PASSED
Issues: []
--------------------------------------------------
7. Develop user interfaces
Validation: PASSED
Issues: []
--------------------------------------------------
8. Conduct pilot testing
Validation: PASSED
Issues: []
--------------------------------------------------
9. Evaluate performance and refine system
Validation: PASSED
Issues: []


In [16]:
def reflect_on_task(task, content):

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": """
                You are a strict quality reviewer for AI-generated
                project proposal content.

                Review the submitted section and identify weaknesses.
                Do not assume proposed targets are verified results.

                Return:
                1. Quality score out of 10
                2. Relevance assessment
                3. Completeness assessment
                4. Technical consistency assessment
                5. Unsupported assumptions or claims
                6. Specific improvement suggestions

                Be objective and concise.
                """
            },
            {
                "role": "user",
                "content": f"""
                Task: {task.title}

                Task Description: {task.description}

                Generated Content:
                {content}

                Review this content carefully.
                """
            }
        ],
        temperature=0.2,
        max_completion_tokens=1200
    )

    return response.choices[0].message.content

In [17]:
reflection = reflect_on_task(
    agent_plan.tasks[0],
    task_results[0].content
)

print("REFLECTION REPORT")
print("=" * 60)
print(reflection)

REFLECTION REPORT
**1. Quality Score (out of 10):** **7.5**  
Well‑structured, clear tables and language; however, several quantitative targets are presented without evidence or a realistic justification, which lowers the overall rigor.

---

**2. Relevance Assessment:** **High**  
The content directly addresses the requested deliverable – definition of project objectives, success criteria, expected outcomes, and stakeholder alignment for an AI‑powered attendance system.

---

**3. Completeness Assessment:** **Medium‑High**  
All typical elements of an objectives section are present (goals, metrics, outcomes, stakeholder mapping). Missing are:  

* Baseline figures against which the targets will be measured.  
* Explicit risk or constraint statements (e.g., hardware limits, network bandwidth).  
* A brief note on how the objectives will be phased or prioritized over the project timeline.

---

**4. Technical Consistency Assessment:** **Medium**  
The goals, success criteria, and outcom

In [18]:
def revise_task_content(task, original_content, reflection):

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": """
                You are an expert technical proposal editor.

                Improve the submitted content using the reviewer feedback.

                Requirements:
                - Address the identified weaknesses.
                - Preserve relevant original information.
                - Remove or qualify unsupported claims.
                - Clearly distinguish assumptions from verified facts.
                - Maintain professional formatting.
                - Do not invent statistics or research evidence.
                - Return only the complete revised section.
                """
            },
            {
                "role": "user",
                "content": f"""
                Task: {task.title}

                Original Content:
                {original_content}

                Reflection Report:
                {reflection}

                Generate the improved content.
                """
            }
        ],
        temperature=0.2,
        max_completion_tokens=1800
    )

    return response.choices[0].message.content

In [19]:
revised_content = revise_task_content(
    agent_plan.tasks[0],
    task_results[0].content,
    reflection
)

print("REVISED CONTENT")
print("=" * 60)
print(revised_content)

REVISED CONTENT
## 1. Project Objectives  

The **AI‑Powered Student Attendance System** will modernize attendance capture, verification, and analysis for educational institutions. The objectives below define the purpose of the solution, the measurable criteria that will indicate success, and the tangible outcomes expected by stakeholders. All quantitative targets are expressed as **targets** that will be validated through a phased pilot and ongoing monitoring; where baseline data are not yet available, the assumptions are explicitly noted.

---

### 1.1 Primary Goals  

| # | Goal | Description | Baseline / Assumption |
|---|------|-------------|-----------------------|
| **1** | **Automated, Real‑Time Attendance Capture** | Use computer‑vision and facial‑recognition models to record student presence instantly as they enter the classroom, replacing manual roll‑calls. | Current process: manual roll‑call averaging **2 min / class** (baseline from school survey). |
| **2** | **High Accur

In [20]:
def execute_and_improve_task(task, document_title):

    # Step 1: Generate initial content
    result = execute_task(task, document_title)

    if result.status != "completed":
        return result, None

    # Step 2: Reflect on generated content
    reflection = reflect_on_task(
        task,
        result.content
    )

    # Step 3: Extract quality score
    import re

    score_match = re.search(
        r"Quality Score.*?(\d+(?:\.\d+)?)",
        reflection,
        re.IGNORECASE
    )

    score = float(score_match.group(1)) if score_match else None

    # Step 4: Revise if quality is below 8
    if score is not None and score < 8:

        print(f"Task {task.task_id}: Revising content...")

        improved_content = revise_task_content(
            task,
            result.content,
            reflection
        )

        result.content = improved_content

    return result, reflection

In [21]:
test_result, test_reflection = execute_and_improve_task(
    agent_plan.tasks[0],
    agent_plan.document_title
)

print("Final Status:", test_result.status)
print("Final Content Length:", len(test_result.content or ""))
print("Reflection Available:", test_reflection is not None)

Final Status: completed
Final Content Length: 5184
Reflection Available: True


In [22]:
def improve_all_tasks(agent_plan, task_results):

    improved_results = []
    reflection_reports = []

    for task, result in zip(agent_plan.tasks, task_results):

        print(f"Reviewing Task {task.task_id}: {task.title}")

        # 1. Reflect on existing content
        reflection = reflect_on_task(
            task,
            result.content
        )

        # 2. Extract quality score
        import re

        match = re.search(
            r"Quality Score.*?(\d+(?:\.\d+)?)",
            reflection,
            re.IGNORECASE
        )

        score = float(match.group(1)) if match else None

        # 3. Revise if score is below 8
        if score is not None and score < 8:

            print(f"Quality score: {score}/10 - Revising")

            result.content = revise_task_content(
                task,
                result.content,
                reflection
            )

        else:
            print(f"Quality score: {score} - No revision required")

        improved_results.append(result)
        reflection_reports.append({
            "task_id": task.task_id,
            "quality_score": score,
            "reflection": reflection
        })

        print("Completed")
        print("-" * 50)

    return improved_results, reflection_reports

In [23]:
final_results, reflection_reports = improve_all_tasks(
    agent_plan,
    task_results
)

Reviewing Task 1: Define project objectives
Quality score: 10.0 - No revision required
Completed
--------------------------------------------------
Reviewing Task 2: Conduct stakeholder analysis
Quality score: 10.0 - No revision required
Completed
--------------------------------------------------
Reviewing Task 3: Design system architecture
Quality score: 10.0 - No revision required
Completed
--------------------------------------------------
Reviewing Task 4: Develop AI model for facial recognition
Quality score: 10.0 - No revision required
Completed
--------------------------------------------------
Reviewing Task 5: Integrate hardware and software components
Quality score: 10.0 - No revision required
Completed
--------------------------------------------------
Reviewing Task 6: Implement privacy and data security measures
Quality score: 10.0 - No revision required
Completed
--------------------------------------------------
Reviewing Task 7: Develop user interfaces
Quality score: 1

In [24]:
final_validation = validate_task_results(final_results)

passed = sum(
    1 for item in final_validation
    if item["passed"]
)

failed = len(final_validation) - passed

print("FINAL QUALITY VALIDATION")
print("=" * 50)
print("Total Tasks:", len(final_results))
print("Passed:", passed)
print("Failed:", failed)

for item in final_validation:

    print(
        f"{item['task_id']}. {item['title']} - "
        f"{'PASSED' if item['passed'] else 'FAILED'}"
    )

    print("Content Length:", item["content_length"])
    print("Issues:", item["issues"])
    print("-" * 50)

FINAL QUALITY VALIDATION
Total Tasks: 10
Passed: 10
Failed: 0
1. Define project objectives - PASSED
Content Length: 5008
Issues: []
--------------------------------------------------
2. Conduct stakeholder analysis - PASSED
Content Length: 5474
Issues: []
--------------------------------------------------
3. Design system architecture - PASSED
Content Length: 4175
Issues: []
--------------------------------------------------
4. Develop AI model for facial recognition - PASSED
Content Length: 4585
Issues: []
--------------------------------------------------
5. Integrate hardware and software components - PASSED
Content Length: 4218
Issues: []
--------------------------------------------------
6. Implement privacy and data security measures - PASSED
Content Length: 4910
Issues: []
--------------------------------------------------
7. Develop user interfaces - PASSED
Content Length: 4827
Issues: []
--------------------------------------------------
8. Conduct pilot testing - PASSED
Conte

In [25]:
total_characters = sum(
    len(result.content or "")
    for result in final_results
)

print("Total Sections:", len(final_results))
print("Total Characters:", total_characters)
print(
    "All Tasks Completed:",
    all(result.status == "completed" for result in final_results)
)

Total Sections: 10
Total Characters: 47107
All Tasks Completed: True


In [26]:
document_sections = []

for result in final_results:

    document_sections.append({
        "heading": result.title,
        "content": result.content
    })

print("Document Title:", agent_plan.document_title)
print("Total Sections:", len(document_sections))

for section in document_sections:
    print("✓", section["heading"])

Document Title: AI Student Attendance System
Total Sections: 10
✓ Define project objectives
✓ Conduct stakeholder analysis
✓ Design system architecture
✓ Develop AI model for facial recognition
✓ Integrate hardware and software components
✓ Implement privacy and data security measures
✓ Develop user interfaces
✓ Conduct pilot testing
✓ Evaluate performance and refine system
✓ Prepare deployment and training plan


In [48]:
import sys

sys.path.insert(0, str(project_root / "app"))

from document_generator import generate_document

output_file = project_root / "outputs" / "AI_Student_Attendance_Proposal.docx"

generated_file = generate_document(
    agent_plan.document_title,
    document_sections,
    output_file
)

print("Document generated successfully!")
print("Saved at:", generated_file)
print("File size:", generated_file.stat().st_size, "bytes")

Document generated successfully!
Saved at: d:\Fluid AI project\Autonomous-AI-Agent\outputs\AI_Student_Attendance_Proposal.docx
File size: 61211 bytes


In [49]:
import importlib
import document_generator

importlib.reload(document_generator)

from document_generator import generate_document

print("Module reloaded successfully!")

Module reloaded successfully!


In [50]:
output_file = project_root / "outputs" / "AI_Student_Attendance_Proposal.docx"

generated_file = generate_document(
    agent_plan.document_title,
    document_sections,
    output_file
)

print("Generated:", generated_file)
print("Exists:", generated_file.exists())
print("File size:", generated_file.stat().st_size, "bytes")

Generated: d:\Fluid AI project\Autonomous-AI-Agent\outputs\AI_Student_Attendance_Proposal.docx
Exists: True
File size: 61216 bytes


In [51]:
import inspect

functions = [
    "generate_plan",
    "execute_task",
    "execute_all_tasks",
    "validate_task_results",
    "reflect_on_task",
    "revise_task_content",
    "improve_all_tasks",
]

for name in functions:
    obj = globals().get(name)
    print(f"{name}: {'Available' if callable(obj) else 'Not found'}")

generate_plan: Not found
execute_task: Available
execute_all_tasks: Available
validate_task_results: Available
reflect_on_task: Available
revise_task_content: Available
improve_all_tasks: Available


In [52]:
import json

def generate_plan(user_request):

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": """
                You are an autonomous AI project planner.
                Break the user's request into clear, executable tasks.

                Return ONLY valid JSON in this structure:
                {
                    "document_type": "string",
                    "document_title": "string",
                    "tasks": [
                        {
                            "task_id": 1,
                            "title": "string",
                            "description": "string",
                            "status": "pending"
                        }
                    ]
                }

                Generate 5 to 12 relevant tasks.
                Do not include Markdown or explanations outside JSON.
                """
            },
            {
                "role": "user",
                "content": user_request
            }
        ],
        temperature=0.2,
        max_completion_tokens=2000
    )

    raw_plan = response.choices[0].message.content

    plan_data = json.loads(raw_plan)

    agent_plan = AgentPlan.model_validate(plan_data)

    return agent_plan

In [53]:
test_request = """
Create a professional project proposal for an AI-powered
student attendance management system.
Include objectives, architecture, implementation,
budget, risks and expected outcomes.
"""

test_plan = generate_plan(test_request)

print("Document:", test_plan.document_title)

for task in test_plan.tasks:
    print(task.task_id, task.title)

Document: AI-Powered Student Attendance Management System
1 Define Project Objectives
2 Research Existing Solutions
3 Design System Architecture
4 Develop Implementation Plan
5 Prepare Budget Estimate
6 Identify Risks and Mitigation Strategies
7 Define Expected Outcomes and KPIs
8 Draft Full Proposal Document
9 Internal Review and Revision
10 Stakeholder Presentation Preparation
11 Finalize and Submit Proposal


In [54]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.agent import (
    generate_plan,
    execute_all_tasks,
    validate_task_results,
    improve_all_tasks
)

print("Agent module imported successfully!")

Agent module imported successfully!


In [55]:
test_request = """
Create a project proposal for an AI-powered student
attendance management system. Include objectives,
scope, technology stack, timeline, budget and risks.
"""

test_plan = generate_plan(test_request)

print("Document:", test_plan.document_title)
print("Number of tasks:", len(test_plan.tasks))

for task in test_plan.tasks:
    print(f"{task.task_id}. {task.title}")

Document: AI-Powered Student Attendance Management System Proposal
Number of tasks: 9
1. Define Project Objectives
2. Outline Project Scope
3. Select Technology Stack
4. Develop Project Timeline
5. Estimate Budget
6. Identify Risks and Mitigation Strategies
7. Draft Proposal Sections
8. Review and Revise Proposal
9. Finalize and Submit Proposal


In [9]:
from app.agent import TaskResult, validate_task_results

test_results = [
    TaskResult(
        task_id=1,
        title="Project Objectives",
        status="completed",
        content="This is a complete sample response." * 15,
        finish_reason="stop"
    ),
    TaskResult(
        task_id=2,
        title="Project Budget",
        status="completed",
        content="This is a truncated sample response." * 15,
        finish_reason="length"
    )
]

report = validate_task_results(test_results)

for item in report:
    print(f"\nTask: {item['title']}")
    print(f"Checks: {item['checks']}")
    print(f"Passed: {item['passed']}")


Task: Project Objectives
Checks: {'completed': True, 'content_exists': True, 'minimum_length': True}
Passed: True

Task: Project Budget
Checks: {'completed': True, 'content_exists': True, 'minimum_length': True}
Passed: True


In [4]:
import os
import sys

print("Current working directory:", os.getcwd())
print("Python executable:", sys.executable)

Current working directory: d:\Fluid AI project\Autonomous-AI-Agent\notebooks
Python executable: d:\Fluid AI project\Autonomous-AI-Agent\autonomous\Scripts\python.exe


In [5]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

os.chdir(PROJECT_ROOT)

print("Project root:", Path.cwd())

Project root: d:\Fluid AI project\Autonomous-AI-Agent


In [6]:
from app.agent import TaskResult, validate_task_results

print("Agent modules imported successfully!")

Agent modules imported successfully!


In [8]:
import importlib
import app.agent

importlib.reload(app.agent)

from app.agent import TaskResult, validate_task_results

In [10]:
import app.agent
import inspect

print("Loaded file:")
print(app.agent.__file__)

print("\nCurrently loaded validation function:")
print(inspect.getsource(app.agent.validate_task_results))

Loaded file:
d:\Fluid AI project\Autonomous-AI-Agent\app\agent.py

Currently loaded validation function:
def validate_task_results(
    task_results: list[TaskResult]
) -> list[dict]:

    validation_report = []

    for result in task_results:
        checks = {
            "completed": result.status == "completed",
            "content_exists": bool(
                result.content and result.content.strip()
            ),
            "minimum_length": bool(
                result.content and len(result.content.strip()) >= 300
            )
        }

        validation_report.append({
            "task_id": result.task_id,
            "title": result.title,
            "checks": checks,
            "passed": all(checks.values())
        })

    return validation_report



In [13]:
import importlib
import app.agent

importlib.reload(app.agent)

from app.agent import TaskResult, validate_task_results

In [1]:
from pathlib import Path
import os

# Ensure we are in the project root
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

from app.document_generator import generate_document

# Sample document sections
sections = [
    {
        "heading": "Project Overview",
        "content": """
        This is a sample AI project proposal.

        The system aims to improve operational efficiency
        using artificial intelligence.
        """
    },
    {
        "heading": "Project Objectives",
        "content": """
        - Improve efficiency.
        - Reduce manual work.
        - Generate useful insights.
        """
    }
]

# Generate test document
output_path = Path("outputs/document_test.docx")

result = generate_document(
    title="AI Project Proposal",
    sections=sections,
    output_path=output_path
)

print("Document generated:", result)
print("File exists:", result.exists())
print("File size:", result.stat().st_size, "bytes")

Document generated: outputs\document_test.docx
File exists: True
File size: 38023 bytes
